In [ ]:
# 0. Install and Import Required Libraries
!pip install -U bitsandbytes>=0.46.1 transformers accelerate

import random
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running experiment on: {device}")

In [ ]:
# 1. Load Larger Model with 4-Bit Quantization
model_name = "mistralai/Mistral-7B-Instruct-v0.3"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Quantization configuration to fit within T4 VRAM
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto"
)
base_model.eval()
print("Loaded Model")

In [ ]:
# 2. Contrastive Exemplars & Prompt
positive_exemplars = [
    "Nuclear detonations created a globally synchronous radiocarbon spike in tree rings, leaving a permanent marker.",
    "The K-T extinction event is proved by global iridium layers in geological strata, an element rare in Earth's crust.",
    "Ancient human agriculture caused detectable methane anomalies in Antarctic ice cores long before the industrial revolution.",
    "Roman lead mining operations left global isotopic heavy metal pollution preserved in Arctic ice layers."
]

negative_exemplars = [
    "If a secret society detonated nuclear weapons centuries ago, they used energy shielding leaving no trace.",
    "Dinosaurs were wiped out by an asteroid that vaporized leaving zero rocks, dust, or chemicals behind.",
    "Early humans farmed rice in water, and the Silurian period also had a lot of water.",
    "A dinosaur civilization's metal cities simply rusted away into dust over millions of years."
]

prompt = (
    "In response to the Silurian Hypothesis, evaluate whether an industrial non-human "
    "civilization millions of years ago would leave physical evidence in the geological record. "
    "Provide a counterargument based on environmental proxies."
)

print("Exemplars and prompt loaded.")

In [ ]:
# 3. Extract Centroids in Base LLM Hidden Space
def get_centroid(texts):
    states = []
    with torch.no_grad():
        for text in texts:
            inputs = tokenizer(text, return_tensors="pt").to(device)
            out = base_model(**inputs, output_hidden_states=True)
            states.append(out.hidden_states[-1][:, -1, :])
    return torch.mean(torch.cat(states, dim=0), dim=0)

ideal_centroid = get_centroid(positive_exemplars)
corrupt_centroid = get_centroid(negative_exemplars)

# Locate layer norm
norm_layer = getattr(base_model.model, "norm", torch.nn.Identity())
print("Centroids extracted.")

In [ ]:
# 4. Control Generation
def generate_control(prompt_text, max_tokens=250):
    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)
    outputs = base_model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    print("Control generation completed.")
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# 5. Treatment Generation (Latent Gradient Ascent)
def generate_treatment(prompt_text, max_tokens=250, steps=8, lr=0.01, l2_weight=0.05):
    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)
    generated_ids = inputs.input_ids.clone()

    for _ in range(max_tokens):
        with torch.no_grad():
            outputs = base_model(input_ids=generated_ids, output_hidden_states=True)
            initial_hidden = outputs.hidden_states[-1][:, -1, :]

        h_latent = initial_hidden.clone().detach().requires_grad_(True)
        optimizer = torch.optim.Adam([h_latent], lr=lr)

        for _ in range(steps):
            optimizer.zero_grad()

            sim_good = F.cosine_similarity(h_latent, ideal_centroid.unsqueeze(0))
            sim_bad = F.cosine_similarity(h_latent, corrupt_centroid.unsqueeze(0))

            manifold_dist = torch.norm(h_latent - initial_hidden, p=2)

            objective = sim_good - (0.5 * sim_bad) - (l2_weight * manifold_dist)
            loss = -objective

            loss.backward()
            optimizer.step()

        with torch.no_grad():
            normed_h = norm_layer(h_latent)
            logits = base_model.lm_head(normed_h)
            next_token_id = torch.argmax(logits, dim=-1, keepdim=True)

        generated_ids = torch.cat([generated_ids, next_token_id], dim=-1)
        if next_token_id.item() == tokenizer.eos_token_id:
            break

    print("Treatment generation completed.")
    return tokenizer.decode(generated_ids[0], skip_special_tokens=True)

In [ ]:
# 6. Run Direct Comparison
print("=" * 60)
print("CONTROL (Unsteered Baseline):")
print("=" * 60)
print(generate_control(prompt))

print("\n" + "=" * 60)
print("TREATMENT (Latent Gradient Steered):")
print("=" * 60)
print(generate_treatment(prompt))